In [1]:
# Construction Progress Monitoring System using Google ADK
# This system leverages Google's Agent Development Kit for multi-agent coordination

import asyncio
import json
import sqlite3
from datetime import datetime, timedelta
from pathlib import Path
from typing import List, Dict, Any, Optional, AsyncGenerator
import base64

from google.adk.agents import LlmAgent, SequentialAgent, BaseAgent
from google.adk.tools import FunctionTool
from google.adk.events import Event, EventActions
from google.adk.agents.invocation_context import InvocationContext
from pydantic import BaseModel, Field


In [ ]:

# =============================================================================
# Data Models
# =============================================================================

class ConstructionElement(BaseModel):
    """Represents a construction element tracked over time"""
    id: str
    name: str
    type: str  # e.g., "beam", "column", "wall", "foundation"
    sector: str  # e.g., "A1", "B2"
    status: str  # e.g., "planned", "in_progress", "completed"
    last_updated: datetime
    coordinates: Optional[Dict[str, float]] = None  # x, y positions in image
    metadata: Dict[str, Any] = Field(default_factory=dict)

class DailyProgress(BaseModel):
    """Represents daily progress data"""
    date: datetime
    images_analyzed: List[str]
    changes_detected: List[Dict[str, Any]]
    elements_updated: List[str]
    progress_percentage: float
    report_summary: str

class ProjectMemory:
    """Project memory system using SQLite for persistence"""
    
    def __init__(self, db_path: str = "construction_memory.db"):
        self.db_path = db_path
        self.init_database()
    
    def init_database(self):
        """Initialize the project memory database"""
        conn = sqlite3.connect(self.db_path)
        cursor = conn.cursor()
        
        # Create construction elements table
        cursor.execute("""
            CREATE TABLE IF NOT EXISTS construction_elements (
                id TEXT PRIMARY KEY,
                name TEXT NOT NULL,
                type TEXT NOT NULL,
                sector TEXT NOT NULL,
                status TEXT NOT NULL,
                last_updated TIMESTAMP NOT NULL,
                coordinates TEXT,
                metadata TEXT
            )
        """)
        
        # Create daily progress table
        cursor.execute("""
            CREATE TABLE IF NOT EXISTS daily_progress (
                date TEXT PRIMARY KEY,
                images_analyzed TEXT,
                changes_detected TEXT,
                elements_updated TEXT,
                progress_percentage REAL,
                report_summary TEXT
            )
        """)
        
        # Create image analysis history
        cursor.execute("""
            CREATE TABLE IF NOT EXISTS image_history (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                date TEXT NOT NULL,
                image_path TEXT NOT NULL,
                analysis_result TEXT,
                timestamp TIMESTAMP DEFAULT CURRENT_TIMESTAMP
            )
        """)
        
        conn.commit()
        conn.close()
    
    def store_element(self, element: ConstructionElement):
        """Store or update a construction element"""
        conn = sqlite3.connect(self.db_path)
        cursor = conn.cursor()
        
        cursor.execute("""
            INSERT OR REPLACE INTO construction_elements 
            (id, name, type, sector, status, last_updated, coordinates, metadata)
            VALUES (?, ?, ?, ?, ?, ?, ?, ?)
        """, (
            element.id, element.name, element.type, element.sector,
            element.status, element.last_updated,
            json.dumps(element.coordinates) if element.coordinates else None,
            json.dumps(element.metadata)
        ))
        
        conn.commit()
        conn.close()
    
    def get_elements_by_sector(self, sector: str) -> List[ConstructionElement]:
        """Retrieve elements by sector"""
        conn = sqlite3.connect(self.db_path)
        cursor = conn.cursor()
        
        cursor.execute("""
            SELECT id, name, type, sector, status, last_updated, coordinates, metadata
            FROM construction_elements WHERE sector = ?
        """, (sector,))
        
        elements = []
        for row in cursor.fetchall():
            element = ConstructionElement(
                id=row[0], name=row[1], type=row[2], sector=row[3],
                status=row[4], last_updated=datetime.fromisoformat(row[5]),
                coordinates=json.loads(row[6]) if row[6] else None,
                metadata=json.loads(row[7])
            )
            elements.append(element)
        
        conn.close()
        return elements
    
    def store_daily_progress(self, progress: DailyProgress):
        """Store daily progress data"""
        conn = sqlite3.connect(self.db_path)
        cursor = conn.cursor()
        
        cursor.execute("""
            INSERT OR REPLACE INTO daily_progress 
            (date, images_analyzed, changes_detected, elements_updated, 
             progress_percentage, report_summary)
            VALUES (?, ?, ?, ?, ?, ?)
        """, (
            progress.date.isoformat(),
            json.dumps(progress.images_analyzed),
            json.dumps(progress.changes_detected),
            json.dumps(progress.elements_updated),
            progress.progress_percentage,
            progress.report_summary
        ))
        
        conn.commit()
        conn.close()


# =============================================================================
# Utility Functions for Tools
# =============================================================================

async def load_images_from_folder(date: str) -> List[str]:
    """Load images from a specific date folder"""
    date_folder = Path(f"construction_images/{date}")
    if not date_folder.exists():
        return []
    
    image_paths = []
    for ext in ['*.jpg', '*.jpeg', '*.png']:
        image_paths.extend(date_folder.glob(ext))
    
    return [str(path) for path in image_paths]

async def encode_image_base64(image_path: str) -> str:
    """Encode image to base64 for VLM processing"""
    try:
        with open(image_path, 'rb') as img_file:
            return base64.b64encode(img_file.read()).decode('utf-8')
    except Exception as e:
        print(f"Error encoding image {image_path}: {e}")
        return ""

async def get_previous_day_state() -> Dict[str, Any]:
    """Retrieve the previous day's construction state"""
    memory = ProjectMemory()
    conn = sqlite3.connect(memory.db_path)
    cursor = conn.cursor()
    
    cursor.execute("""
        SELECT date, elements_updated, changes_detected 
        FROM daily_progress 
        ORDER BY date DESC LIMIT 1
    """)
    
    result = cursor.fetchone()
    conn.close()
    
    if result:
        return {
            "date": result[0],
            "elements": json.loads(result[1]),
            "changes": json.loads(result[2])
        }
    return {}


# =============================================================================
# Custom Agents
# =============================================================================

class ImageAnalysisAgent(BaseAgent):
    """Agent specialized in analyzing construction images using VLM"""
    
    def __init__(self):
        super().__init__(
            name="ImageAnalyzer",
            description="Analyzes construction site images to identify elements and their status"
        )
    
    async def _run_async_impl(self, ctx: InvocationContext) -> AsyncGenerator[Event, None]:
        """Analyze images and extract construction elements"""
        current_date = ctx.session.state.get("current_date", datetime.now().isoformat()[:10])
        image_paths = await load_images_from_folder(current_date)
        
        if not image_paths:
            yield Event(
                author=self.name,
                content=f"No images found for date {current_date}",
                actions=EventActions(escalate=True)
            )
            return
        
        analysis_results = []
        for image_path in image_paths:
            # In a real implementation, this would call a VLM API
            # For now, we'll simulate the analysis
            analysis = {
                "image_path": image_path,
                "elements_detected": [
                    {
                        "id": f"beam_A1_{len(analysis_results)}",
                        "name": f"Beam A1-{len(analysis_results)}",
                        "type": "beam",
                        "sector": "A1",
                        "status": "in_progress",
                        "confidence": 0.95,
                        "coordinates": {"x": 100, "y": 200}
                    }
                ],
                "safety_concerns": [],
                "weather_conditions": "clear"
            }
            analysis_results.append(analysis)
        
        # Store results in session state
        ctx.session.state["image_analysis_results"] = analysis_results
        ctx.session.state["images_processed"] = len(image_paths)
        
        yield Event(
            author=self.name,
            content=f"Analyzed {len(image_paths)} images, detected {sum(len(r['elements_detected']) for r in analysis_results)} elements"
        )

class ChangeDetectionAgent(BaseAgent):
    """Agent that compares current day images with previous day state"""
    
    def __init__(self):
        super().__init__(
            name="ChangeDetector", 
            description="Detects changes between current and previous day construction state"
        )
    
    async def _run_async_impl(self, ctx: InvocationContext) -> AsyncGenerator[Event, None]:
        """Detect changes from previous day"""
        current_results = ctx.session.state.get("image_analysis_results", [])
        previous_state = await get_previous_day_state()
        
        changes_detected = []
        new_elements = []
        updated_elements = []
        
        for result in current_results:
            for element in result["elements_detected"]:
                # Simple change detection logic (in real implementation, would be more sophisticated)
                element_id = element["id"]
                
                # Check if this is a new element
                if not any(elem_id == element_id for elem_id in previous_state.get("elements", [])):
                    new_elements.append(element)
                    changes_detected.append({
                        "type": "new_element",
                        "element_id": element_id,
                        "description": f"New {element['type']} detected in sector {element['sector']}"
                    })
                else:
                    # Check for status changes
                    updated_elements.append(element)
                    changes_detected.append({
                        "type": "status_update", 
                        "element_id": element_id,
                        "description": f"{element['type']} {element['name']} status updated to {element['status']}"
                    })
        
        ctx.session.state["changes_detected"] = changes_detected
        ctx.session.state["new_elements"] = new_elements
        ctx.session.state["updated_elements"] = updated_elements
        
        yield Event(
            author=self.name,
            content=f"Detected {len(changes_detected)} changes: {len(new_elements)} new elements, {len(updated_elements)} updates"
        )

class MemoryUpdateAgent(BaseAgent):
    """Agent that updates the project memory system"""
    
    def __init__(self):
        super().__init__(
            name="MemoryUpdater",
            description="Updates the project memory database with new construction state"
        )
        self.memory = ProjectMemory()
    
    async def _run_async_impl(self, ctx: InvocationContext) -> AsyncGenerator[Event, None]:
        """Update project memory with current analysis"""
        new_elements = ctx.session.state.get("new_elements", [])
        updated_elements = ctx.session.state.get("updated_elements", [])
        changes = ctx.session.state.get("changes_detected", [])
        
        elements_stored = 0
        
        # Store new elements
        for element_data in new_elements:
            element = ConstructionElement(
                id=element_data["id"],
                name=element_data["name"],
                type=element_data["type"],
                sector=element_data["sector"],
                status=element_data["status"],
                last_updated=datetime.now(),
                coordinates=element_data.get("coordinates"),
                metadata={"confidence": element_data.get("confidence", 0.0)}
            )
            self.memory.store_element(element)
            elements_stored += 1
        
        # Update existing elements
        for element_data in updated_elements:
            element = ConstructionElement(
                id=element_data["id"],
                name=element_data["name"], 
                type=element_data["type"],
                sector=element_data["sector"],
                status=element_data["status"],
                last_updated=datetime.now(),
                coordinates=element_data.get("coordinates"),
                metadata={"confidence": element_data.get("confidence", 0.0)}
            )
            self.memory.store_element(element)
            elements_stored += 1
        
        ctx.session.state["elements_stored"] = elements_stored
        
        yield Event(
            author=self.name,
            content=f"Updated project memory: stored {elements_stored} elements"
        )


# =============================================================================
# Tools for LLM Agents
# =============================================================================

async def calculate_progress_percentage(sector: str = "all") -> float:
    """Calculate overall project progress percentage"""
    memory = ProjectMemory()
    conn = sqlite3.connect(memory.db_path)
    cursor = conn.cursor()
    
    if sector == "all":
        cursor.execute("SELECT status FROM construction_elements")
    else:
        cursor.execute("SELECT status FROM construction_elements WHERE sector = ?", (sector,))
    
    elements = cursor.fetchall()
    conn.close()
    
    if not elements:
        return 0.0
    
    completed = sum(1 for elem in elements if elem[0] == "completed")
    return (completed / len(elements)) * 100

async def get_project_timeline_forecast() -> Dict[str, Any]:
    """Generate project timeline forecast based on current progress"""
    progress = await calculate_progress_percentage()
    
    # Simple linear projection (in real implementation would be more sophisticated)
    if progress > 0:
        days_elapsed = 30  # Placeholder
        total_days_estimate = (days_elapsed / progress) * 100
        days_remaining = total_days_estimate - days_elapsed
    else:
        days_remaining = 365  # Default estimate
    
    completion_date = datetime.now() + timedelta(days=days_remaining)
    
    return {
        "current_progress": progress,
        "estimated_completion": completion_date.isoformat(),
        "days_remaining": int(days_remaining),
        "confidence": "medium"
    }

# Create tools
progress_tool = FunctionTool(func=calculate_progress_percentage)
forecast_tool = FunctionTool(func=get_project_timeline_forecast)


# =============================================================================
# LLM Agents
# =============================================================================

# Progress Analysis Agent
progress_analyzer = LlmAgent(
    name="ProgressAnalyzer",
    model="gemini-2.0-flash",
    instruction="""
    You are a construction progress analyst. Analyze the current construction state and provide insights.
    
    Your tasks:
    1. Review the changes detected from image analysis
    2. Calculate progress metrics using the progress tool
    3. Generate timeline forecasts using the forecast tool
    4. Identify any potential delays or issues
    5. Save your analysis summary to state key 'progress_analysis'
    
    Be specific about progress percentages, timeline estimates, and any concerns.
    """,
    tools=[progress_tool, forecast_tool],
    output_key="progress_analysis"
)

# Report Generator Agent
report_generator = LlmAgent(
    name="ReportGenerator", 
    model="gemini-2.0-flash",
    instruction="""
    You are a construction reporting specialist. Generate a comprehensive daily progress report.
    
    Use the following information from the session state:
    - Image analysis results
    - Changes detected
    - Progress analysis
    - Elements stored in memory
    
    Generate a detailed, professional report that includes:
    1. Executive Summary
    2. Daily Changes and Progress
    3. Construction Elements Status
    4. Timeline and Forecasts
    5. Issues and Recommendations
    6. Visual Summary (description of key images)
    
    Make the report actionable for project managers and stakeholders.
    Save the final report to state key 'daily_report'.
    """,
    output_key="daily_report"
)

# Safety Monitor Agent
safety_monitor = LlmAgent(
    name="SafetyMonitor",
    model="gemini-2.0-flash", 
    instruction="""
    You are a construction safety specialist. Review the image analysis results for safety concerns.
    
    Look for:
    1. Safety equipment compliance
    2. Hazardous conditions
    3. Proper construction practices
    4. Emergency access routes
    5. Weather-related safety issues
    
    If you identify any safety concerns, escalate them immediately and save details to state key 'safety_concerns'.
    If no concerns, save "No safety issues detected" to the same key.
    """,
    output_key="safety_concerns"
)


# =============================================================================
# Main Coordinator Agent and Workflow
# =============================================================================

# Coordinator Agent with LLM-driven delegation
coordinator = LlmAgent(
    name="ProjectCoordinator",
    model="gemini-2.0-flash",
    instruction="""
    You are the main construction project coordinator. Your role is to orchestrate the daily analysis workflow.
    
    Available sub-agents:
    - ProgressAnalyzer: Analyzes progress metrics and forecasts
    - ReportGenerator: Creates comprehensive daily reports  
    - SafetyMonitor: Monitors safety compliance and issues
    
    Coordinate these agents to produce a complete daily analysis. Ensure all agents complete their tasks
    before finalizing the daily workflow.
    """,
    description="Main project coordinator for construction monitoring",
    sub_agents=[progress_analyzer, report_generator, safety_monitor]
)

# Complete workflow using Sequential Pipeline
construction_monitoring_workflow = SequentialAgent(
    name="ConstructionMonitoringWorkflow",
    sub_agents=[
        ImageAnalysisAgent(),         # 1. Analyze images
        ChangeDetectionAgent(),       # 2. Detect changes  
        MemoryUpdateAgent(),         # 3. Update memory
        coordinator                   # 4. Generate insights and reports
    ]
)


# =============================================================================
# Main Application
# =============================================================================

class ConstructionMonitoringApp:
    """Main application class for the construction monitoring system"""
    
    def __init__(self):
        self.workflow = construction_monitoring_workflow
        self.memory = ProjectMemory()
    
    async def run_daily_analysis(self, date: str = None) -> Dict[str, Any]:
        """Run the complete daily analysis workflow"""
        if date is None:
            date = datetime.now().isoformat()[:10]
        
        print(f"Starting daily analysis for {date}...")
        
        # Initialize session context
        from google.adk.core import Session
        session = Session()
        session.state["current_date"] = date
        
        # Create invocation context
        ctx = InvocationContext(session=session)
        
        # Run the workflow
        results = []
        async for event in self.workflow._run_async_impl(ctx):
            results.append(event)
            if hasattr(event, 'content') and event.content:
                print(f"[{event.author}]: {event.content}")
        
        # Extract final results
        final_results = {
            "date": date,
            "images_processed": session.state.get("images_processed", 0),
            "changes_detected": session.state.get("changes_detected", []),
            "progress_analysis": session.state.get("progress_analysis", ""),
            "daily_report": session.state.get("daily_report", ""),
            "safety_concerns": session.state.get("safety_concerns", ""),
            "elements_stored": session.state.get("elements_stored", 0)
        }
        
        # Store daily progress
        progress = DailyProgress(
            date=datetime.fromisoformat(date),
            images_analyzed=session.state.get("image_analysis_results", []),
            changes_detected=session.state.get("changes_detected", []),
            elements_updated=session.state.get("updated_elements", []),
            progress_percentage=await calculate_progress_percentage(),
            report_summary=session.state.get("daily_report", "")
        )
        self.memory.store_daily_progress(progress)
        
        return final_results
    
    async def get_project_status(self) -> Dict[str, Any]:
        """Get current project status"""
        progress = await calculate_progress_percentage()
        forecast = await get_project_timeline_forecast()
        
        return {
            "overall_progress": progress,
            "timeline_forecast": forecast,
            "last_analysis": datetime.now().isoformat()
        }


# =============================================================================
# Example Usage
# =============================================================================

async def main():
    """Example usage of the construction monitoring system"""
    app = ConstructionMonitoringApp()
    
    # Run daily analysis
    results = await app.run_daily_analysis("2025-06-18")
    
    print("\n" + "="*80)
    print("DAILY ANALYSIS RESULTS")
    print("="*80)
    print(f"Date: {results['date']}")
    print(f"Images Processed: {results['images_processed']}")
    print(f"Changes Detected: {len(results['changes_detected'])}")
    print(f"Elements Stored: {results['elements_stored']}")
    print(f"\nProgress Analysis:\n{results['progress_analysis']}")
    print(f"\nSafety Concerns:\n{results['safety_concerns']}")
    print(f"\nDaily Report:\n{results['daily_report']}")
    
    # Get project status
    status = await app.get_project_status()
    print(f"\nProject Status:\n{status}")


if __name__ == "__main__":
    # Install dependencies first:
    # pip install google-adk pydantic sqlite3
    
    asyncio.run(main())